In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

BASE_PATH = "/kaggle/input/competitions/digitrecognition-ee708" 
TRAIN_AUDIO_DIR = os.path.join(BASE_PATH, "train_audio", "train_audio")
TEST_AUDIO_DIR = os.path.join(BASE_PATH, "test_audio", "test_audio")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
SAMPLE_SUB_CSV = os.path.join(BASE_PATH, "sample_submission.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15
LR = 0.001
SAMPLE_RATE = 16000
MAX_LEN = 16000
N_MELS = 64

In [2]:
class DigitAudioDataset(Dataset):
    def __init__(self, csv_file, audio_dir, is_test=False):
        self.audio_dir = audio_dir
        self.is_test = is_test
        
        if self.is_test:
            test_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
            if not test_files:
                raise ValueError(f"CRITICAL ERROR: No .wav files found in {audio_dir}")
            self.df = pd.DataFrame({'id': [f.replace('.wav', '') for f in test_files]})
            print(f"Test Set: Loaded {len(self.df)} files directly.")
        else:
            temp_df = pd.read_csv(csv_file)
            temp_df['filename'] = temp_df['id'].astype(str).str.strip().apply(lambda x: x if x.endswith('.wav') else f"{x}.wav")
            temp_df['exists'] = temp_df['filename'].apply(lambda x: os.path.exists(os.path.join(audio_dir, x)))
            self.df = temp_df[temp_df['exists']].reset_index(drop=True)
            
            if self.df.empty:
                contents = os.listdir(audio_dir)[:5] if os.path.exists(audio_dir) else "Directory missing!"
                raise ValueError(f"CRITICAL ERROR: 0 files matched! Folder contains: {contents}")
            print(f"Train Set: Kept {len(self.df)} valid files.")
        
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=N_MELS)
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=30)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_id = str(row['id']).strip()
        filename = audio_id if audio_id.endswith('.wav') else f"{audio_id}.wav"
        audio_path = os.path.join(self.audio_dir, filename)
        
        waveform, sr = torchaudio.load(audio_path)
        
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(waveform)
            
        waveform = torch.mean(waveform, dim=0, keepdim=True) if waveform.shape[0] > 1 else waveform
            
        if waveform.shape[1] > MAX_LEN:
            waveform = waveform[:, :MAX_LEN]
        elif waveform.shape[1] < MAX_LEN:
            waveform = F.pad(waveform, (0, MAX_LEN - waveform.shape[1]))
            
        mel_spec = self.amplitude_to_db(self.mel_spectrogram(waveform))
        
        if not self.is_test and torch.rand(1).item() > 0.5:
            mel_spec = self.time_mask(self.freq_mask(mel_spec))
            
        return (mel_spec, audio_id) if self.is_test else (mel_spec, int(row['label']))

train_dataset = DigitAudioDataset(TRAIN_CSV, TRAIN_AUDIO_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = DigitAudioDataset(SAMPLE_SUB_CSV, TEST_AUDIO_DIR, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Train Set: Kept 37800 valid files.
Test Set: Loaded 16200 files directly.


In [3]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class AudioResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.prep = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.layer1 = ResBlock(32, 64, stride=2)
        self.layer2 = ResBlock(64, 128, stride=2)
        self.layer3 = ResBlock(128, 256, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        return self.fc(x)

model = AudioResNet().to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

In [4]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for specs, labels in train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} LR: {scheduler.get_last_lr()[0]:.6f}")

print("\nTraining complete!")
model.eval()
submission_ids = []
submission_labels = []

with torch.no_grad():
    for specs, audio_ids in test_loader:
        specs = specs.to(DEVICE)
        outputs = model(specs)
        _, predicted = torch.max(outputs, 1)
        
        submission_ids.extend(audio_ids)
        submission_labels.extend(predicted.cpu().tolist())

pd.DataFrame({
    'id': submission_ids,
    'label': submission_labels
}).to_csv('submission.csv', index=False)

Epoch [1/15] Loss: 0.9566 LR: 0.000989
Epoch [2/15] Loss: 0.6847 LR: 0.000957
Epoch [3/15] Loss: 0.6505 LR: 0.000905
Epoch [4/15] Loss: 0.6326 LR: 0.000835
Epoch [5/15] Loss: 0.6137 LR: 0.000750
Epoch [6/15] Loss: 0.6084 LR: 0.000655
Epoch [7/15] Loss: 0.5950 LR: 0.000553
Epoch [8/15] Loss: 0.5863 LR: 0.000448
Epoch [9/15] Loss: 0.5780 LR: 0.000346
Epoch [10/15] Loss: 0.5691 LR: 0.000251
Epoch [11/15] Loss: 0.5629 LR: 0.000166
Epoch [12/15] Loss: 0.5557 LR: 0.000096
Epoch [13/15] Loss: 0.5506 LR: 0.000044
Epoch [14/15] Loss: 0.5476 LR: 0.000012
Epoch [15/15] Loss: 0.5458 LR: 0.000001

Training complete!
